# 卫星-经济预测：从准备数据到训练

**用法**：依次点 Run 每个 cell。每个 section 前都标注了用什么 runtime（T4 / V100 / A100）。

**前置**：
1. 代码已 push 到 GitHub（替换下面的 REPO_URL）
2. GEE 导出的数据已在 Drive `gee_guangdong_s2/` 文件夹
3. `guangdong_county_gdp.csv` 已在 repo 的 `data/raw/` 里

## 0. Setup（任意 runtime 都行，T4 最便宜）

In [ ]:
# 0.1 挂载 Drive
from google.colab import drive
drive.mount('/content/drive')

# 0.2 验证 GEE 数据存在
import os
GEE_DIR = '/content/drive/MyDrive/gee_guangdong_s2'
print('GEE folder:', GEE_DIR, 'exists:', os.path.isdir(GEE_DIR))
!ls {GEE_DIR} | head -10
!ls {GEE_DIR} | wc -l

In [ ]:
# 0.3 clone 代码（替换成你的 GitHub repo URL）
REPO_URL = 'https://github.com/你的用户名/satellite-econ-guangdong.git'

%cd /content
!rm -rf /content/proj
!git clone {REPO_URL} /content/proj
%cd /content/proj

# 0.4 装依赖
!pip install -q -r requirements.txt

## 1. 合并 viirs CSV，跑标签处理（T4 即可，~2 分钟）

In [ ]:
import shutil, glob
import pandas as pd
from pathlib import Path

# 1.1 把 grid_meta.csv 和 viirs_*.csv 从 Drive 复制到本地 data/processed/
Path('data/processed').mkdir(parents=True, exist_ok=True)

shutil.copy(f'{GEE_DIR}/grid_meta.csv', 'data/processed/grid_meta.csv')
for f in glob.glob(f'{GEE_DIR}/viirs_*.csv'):
    shutil.copy(f, f'data/processed/{Path(f).name}')

# 1.2 合并 viirs_*.csv
dfs = [pd.read_csv(f) for f in sorted(glob.glob('data/processed/viirs_*.csv'))]
viirs = pd.concat(dfs, ignore_index=True)
viirs.to_csv('data/processed/viirs.csv', index=False)
print(f'viirs merged: {len(viirs)} rows')
print(viirs.head())

# 1.3 查 grid_meta
gm = pd.read_csv('data/processed/grid_meta.csv')
print(f'\ngrid_meta: {len(gm)} rows, county_name distribution:')
print(gm['county_name'].value_counts())

In [ ]:
# 1.4 跑标签处理（用 repo 里自带的 guangdong_county_gdp.csv）
!python scripts/02_prepare_labels.py

# 检查产物
!ls -la data/processed/
import json
split = json.load(open('data/processed/split.json'))
for k, v in split.items():
    print(f'  {k}: {len(v)} unique grid_ids')

## 2. 一次性把 GeoTIFF 打包到 Drive（T4，30-60 分钟）

目的：之后每次开新 session，从 Drive 解 tar 到本地 SSD 比 file-by-file 复制快几十倍。**只跑一次**。

如果你已经跑过，跳到 Section 3。

In [ ]:
TAR_PATH = '/content/drive/MyDrive/satellite-econ-tars/all_tifs.tar'

if os.path.exists(TAR_PATH):
    print(f'[跳过] {TAR_PATH} 已存在')
else:
    !mkdir -p {os.path.dirname(TAR_PATH)}
    print('打包 28k tif 到一个 tar（无压缩，速度优先）...')
    # 直接 cd 到 GEE 文件夹后 tar，避免完整路径
    !cd {GEE_DIR} && tar -cf {TAR_PATH} *.tif
    !ls -lh {TAR_PATH}

## 3. 解 tar 到本地 SSD（每次开新 session 必跑）

Colab A100 给 ~200GB 本地 SSD，比 Drive 快几十倍。

In [ ]:
TAR_PATH = '/content/drive/MyDrive/satellite-econ-tars/all_tifs.tar'

!mkdir -p data/sentinel2
print('解 tar 到本地 SSD...')
!tar -xf {TAR_PATH} -C data/sentinel2/
!ls data/sentinel2/ | wc -l

In [ ]:
# 让 checkpoint 直接写到 Drive，session 断了不丢
CKPT_DRIVE = '/content/drive/MyDrive/satellite-econ-checkpoints'
!mkdir -p {CKPT_DRIVE}
!rm -rf outputs/checkpoints && ln -sfn {CKPT_DRIVE} outputs/checkpoints
!ls -la outputs/checkpoints/

## 4. wandb 登录

In [ ]:
!wandb login
# 也可以改 configs/default.yaml 里 wandb.mode 为 'offline' 或 'disabled' 跳过

## 5. 冒烟测试（T4 即可）：M1 跑 2 epoch 看管线

如果这步 OK，再切 A100 跑正式训练。

In [ ]:
!python -m src.train --model M1 --mode level --epochs 2 --batch-size 8

## 6. 正式训练（切 A100：Runtime → Change runtime type → A100）

切完 runtime 要重跑 Section 0、1（如果 labels 不在）、3（解 tar）才能继续。

**每个 run 大约 1-3 小时，结束后立刻关 runtime 省单元。**

In [ ]:
# 6.1 M1 level（基线）
!python -m src.train --model M1 --mode level --batch-size 128

In [ ]:
# 6.2 M2 level（主模型）
!python -m src.train --model M2 --mode level --batch-size 128

In [ ]:
# 6.3 M1 / M2 diff1
!python -m src.train --model M1 --mode diff1 --batch-size 128
!python -m src.train --model M2 --mode diff1 --batch-size 128

In [ ]:
# 6.4 (可选) M3 / M4 —— 切回 V100 节省单元
!python -m src.train --model M3 --mode level --batch-size 64
!python -m src.train --model M3 --mode diff1 --batch-size 64

## 7. 评估和可视化（T4 即可）

In [ ]:
!python -m src.evaluate \
    --checkpoints \
        M1:outputs/checkpoints/M1_level_seed42_best.pt \
        M2:outputs/checkpoints/M2_level_seed42_best.pt \
    --mode level

In [ ]:
!python -m src.visualize summary \
    --results outputs/figures/results_table.csv \
    --predictions outputs/figures/predictions_level.csv

!python -m src.visualize attention \
    --checkpoint outputs/checkpoints/M2_level_seed42_best.pt \
    --n-samples 6

# 同步图到 Drive 备份
!cp -r outputs/figures /content/drive/MyDrive/satellite-econ-figures/